# Differencial Privacy - Stochastic Gradient Descent

## Setup

In [ ]:
pip install --upgrade git+https://github.com/PandaSt0rm/htb-ai-library

In [2]:
import os
import json
import torch
import torch.optim as optim
from safetensors.torch import save_file

from htb_ai_library import (
    set_reproducibility, use_htb_style,
    CIFAR10CNN,
    get_cifar10_loaders,
    train_baseline_sgd,
    train_dp_sgd,
    evaluate_accuracy,
    compute_mia_advantage,
    plot_accuracy_comparison,
    plot_privacy_utility_tradeoff,
)

In [3]:
RANDOM_SEED = 1337
BATCH_SIZE = 256
BASELINE_EPOCHS = 20
BASELINE_LR = 0.1
DP_EPOCHS = 20
DP_LR = 0.1
MAX_GRAD_NORM = 1.0
DELTA = 1e-5

set_reproducibility(RANDOM_SEED)
use_htb_style()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs("figs", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)


## Training Models and Measuring Privacy


In [4]:
print("=" * 80)
print("  DP-SGD PRIVACY MITIGATION DEMONSTRATION")
print("=" * 80)
print(f"\nDevice: {device}")
print(f"Random seed: {RANDOM_SEED}")


print("\nLoading CIFAR-10 dataset...")
train_dataset, test_dataset, train_loader, test_loader = get_cifar10_loaders(batch_size=BATCH_SIZE, download=True)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Batch size: {BATCH_SIZE}")

  DP-SGD PRIVACY MITIGATION DEMONSTRATION

Device: cpu
Random seed: 1337

Loading CIFAR-10 dataset...
Files already downloaded and verified
Files already downloaded and verified
Training samples: 50,000
Test samples: 10,000
Batch size: 256


In [5]:
# Trainig the baseline model
print("\n" + "=" * 80)
print("  TRAINING: BASELINE MODEL (No Privacy Protection)")
print("=" * 80)

baseline_model = CIFAR10CNN().to(device)
baseline_model = train_baseline_sgd(baseline_model, train_loader, device, epochs=BASELINE_EPOCHS, learning_rate=BASELINE_LR)
# Output: Epoch 1/20 - Loss: 2.31, Acc: 25.4%
# Output: Epoch 10/20 - Loss: 1.12, Acc: 61.2%
# Output: Epoch 20/20 - Loss: 0.68, Acc: 77.3%



  TRAINING: BASELINE MODEL (No Privacy Protection)
Epoch [1/20] Loss: 1.6139 | Acc: 40.86%
Epoch [2/20] Loss: 1.1724 | Acc: 58.56%
Epoch [3/20] Loss: 1.0082 | Acc: 64.97%
Epoch [4/20] Loss: 0.9201 | Acc: 68.08%
Epoch [5/20] Loss: 0.8478 | Acc: 70.70%
Epoch [6/20] Loss: 0.7969 | Acc: 72.54%
Epoch [7/20] Loss: 0.7620 | Acc: 73.90%
Epoch [8/20] Loss: 0.7255 | Acc: 75.23%
Epoch [9/20] Loss: 0.7103 | Acc: 75.57%
Epoch [10/20] Loss: 0.6947 | Acc: 76.46%
Epoch [11/20] Loss: 0.7044 | Acc: 76.26%
Epoch [12/20] Loss: 0.6853 | Acc: 77.05%
Epoch [13/20] Loss: 0.6821 | Acc: 77.07%
Epoch [14/20] Loss: 0.6799 | Acc: 77.68%
Epoch [15/20] Loss: 0.6674 | Acc: 78.20%
Epoch [16/20] Loss: 0.6897 | Acc: 77.63%
Epoch [17/20] Loss: 0.6703 | Acc: 78.13%
Epoch [18/20] Loss: 0.7118 | Acc: 77.33%
Epoch [19/20] Loss: 0.6943 | Acc: 77.81%
Epoch [20/20] Loss: 0.7039 | Acc: 77.58%


In [6]:
# evaluating baseline performance

train_acc_baseline = evaluate_accuracy(baseline_model, train_loader, device)
test_acc_baseline = evaluate_accuracy(baseline_model, test_loader, device)

print("\nBaseline Model Performance:")
print(f"  Training accuracy: {train_acc_baseline:.2f}%")
print(f"  Test accuracy: {test_acc_baseline:.2f}%")
print(f"  Overfitting gap: {train_acc_baseline - test_acc_baseline:.2f}%")



Baseline Model Performance:
  Training accuracy: 75.91%
  Test accuracy: 64.95%
  Overfitting gap: 10.96%


In [7]:
torch.save(baseline_model.state_dict(), "output/baseline_model.pth")
print("\nSaved baseline model to output/baseline_model.pth")


Saved baseline model to output/baseline_model.pth


In [8]:
# Measuring Baseline Membership Inference Vulnerability

print("\n" + "=" * 80)
print("  MEMBERSHIP INFERENCE MEASUREMENT: Baseline Model")
print("=" * 80)

mia_acc_baseline, mia_adv_baseline = compute_mia_advantage(
    baseline_model, train_loader, test_loader, device
)

print("\nMIA Results (Baseline):")
print(f"  Attack accuracy: {mia_acc_baseline:.4f}")
print(f"  Attack advantage: {mia_adv_baseline:.4f}")
print("  Random baseline: 0.5000")


  MEMBERSHIP INFERENCE MEASUREMENT: Baseline Model

MIA Results (Baseline):
  Attack accuracy: 0.5224
  Attack advantage: 0.0224
  Random baseline: 0.5000


In [9]:
# defending with DP-SGD

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

In [10]:
# training iwth epsilon = 10
print("\n" + "=" * 80)
print("  TRAINING: DP-SGD MODEL (Target ε=10)")
print("=" * 80)

TARGET_EPSILON_10 = 10.0

_, _, train_loader_dp, test_loader_dp = get_cifar10_loaders(batch_size=BATCH_SIZE, download=False)

dp_model_10 = CIFAR10CNN().to(device)
dp_model_10 = ModuleValidator.fix(dp_model_10)
optimizer_dp = optim.SGD(dp_model_10.parameters(), lr=DP_LR, momentum=0.9)


  TRAINING: DP-SGD MODEL (Target ε=10)


In [ ]:
privacy_engine = PrivacyEngine(accountant="rdp")
dp_model_10, optimizer_dp, train_loader_dp = privacy_engine.make_private_with_epsilon(
    module = dp_model_10,
    optimizer = optimizer_dp,
    data_loader=train_loader_dp,
    target_epsilon=TARGET_EPSILON_10,
    target_delta=DELTA,
    epochs=DP_EPOCHS,
    max_grad_norm=MAX_GRAD_NORM
)

print(f"\nConfiguration:")
print(f"  Target epsilon: {TARGET_EPSILON_10}")
print(f"  Delta: {DELTA}")
print(f"  Max gradient norm: {MAX_GRAD_NORM}")

In [ ]:
dp_model_10, final_epsilon_10 = train_dp_sgd(
    dp_model_10, train_loader_dp, privacy_engine, optimizer_dp, device
)
# Output: Epoch 1/20 - Loss: 2.35, Acc: 22.1%, ε: 1.24
# Output: Epoch 10/20 - Loss: 1.52, Acc: 48.3%, ε: 5.82
# Output: Epoch 20/20 - Loss: 1.21, Acc: 58.4%, ε: 10.00

print(f"\nFinal privacy guarantee: (ε={final_epsilon_10:.2f}, δ={DELTA})")
# Output: Final privacy guarantee: (ε=10.00, δ=1e-05)

train_acc_dp10 = evaluate_accuracy(dp_model_10, train_loader_dp, device)
test_acc_dp10 = evaluate_accuracy(dp_model_10, test_loader_dp, device)

print(f"\nDP Model (ε=10) Performance:")
print(f"  Training accuracy: {train_acc_dp10:.2f}%")
print(f"  Test accuracy: {test_acc_dp10:.2f}%")
print(f"  Overfitting gap: {train_acc_dp10 - test_acc_dp10:.2f}%")
# Output: Training accuracy: 61.24%, Test accuracy: 58.15%, Overfitting gap: 3.09%

In [13]:
torch.save(dp_model_10._module.state_dict(), "output/dp_model_eps10.pth")

# Save in safetensors format for validator API
save_file(dp_model_10._module.state_dict(), "models/dp_model.safetensors")
print("Saved DP model (ε=10) to models/dp_model.safetensors for validation")

Saved DP model (ε=10) to models/dp_model.safetensors for validation


In [14]:
print("\n" + "=" * 80)
print("  MEMBERSHIP INFERENCE MEASUREMENT: DP Model (ε=10)")
print("=" * 80)

mia_acc_dp10, mia_adv_dp10 = compute_mia_advantage(
    dp_model_10, train_loader_dp, test_loader_dp, device
)

print(f"\nMIA Results (DP ε=10):")
print(f"  Attack accuracy: {mia_acc_dp10:.4f}")
print(f"  Attack advantage: {mia_adv_dp10:.4f}")

improvement_10 = mia_adv_baseline - mia_adv_dp10
print(f"\nPrivacy Improvement vs Baseline:")
print(f"  MIA advantage reduction: {improvement_10:+.4f}")
print(f"  Accuracy cost: {test_acc_baseline - test_acc_dp10:.2f}%")



  MEMBERSHIP INFERENCE MEASUREMENT: DP Model (ε=10)

MIA Results (DP ε=10):
  Attack accuracy: 0.5072
  Attack advantage: 0.0072

Privacy Improvement vs Baseline:
  MIA advantage reduction: +0.0152
  Accuracy cost: 6.23%


In [15]:
print("\n" + "=" * 80)
print("  TRAINING: DP-SGD MODEL (Target ε=3)")
print("=" * 80)

TARGET_EPSILON_3 = 3.0

_, _, train_loader_dp3, test_loader_dp3 = get_cifar10_loaders(batch_size=BATCH_SIZE, download=False)

dp_model_3 = CIFAR10CNN().to(device)
dp_model_3 = ModuleValidator.fix(dp_model_3)
optimizer_dp3 = optim.SGD(dp_model_3.parameters(), lr=DP_LR, momentum=0.9)

privacy_engine_3 = PrivacyEngine(accountant="rdp")
dp_model_3, optimizer_dp3, train_loader_dp3 = privacy_engine_3.make_private_with_epsilon(
    module=dp_model_3,
    optimizer=optimizer_dp3,
    data_loader=train_loader_dp3,
    target_epsilon=TARGET_EPSILON_3,
    target_delta=DELTA,
    epochs=DP_EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
)

dp_model_3, final_epsilon_3 = train_dp_sgd(
    dp_model_3, train_loader_dp3, privacy_engine_3, optimizer_dp3, device
)
# Output: Epoch 20/20 - Loss: 1.45, Acc: 53.2%, ε: 3.00

print(f"\nFinal privacy guarantee: (ε={final_epsilon_3:.2f}, δ={DELTA})")
# Output: Final privacy guarantee: (ε=3.00, δ=1e-05)

torch.save(dp_model_3._module.state_dict(), "output/dp_model_eps3.pth")


  TRAINING: DP-SGD MODEL (Target ε=3)
Epoch [1/20] Loss: 1.9593 | Acc: 31.59% | ε: 1.57
Epoch [2/20] Loss: 1.8513 | Acc: 42.92% | ε: 1.69
Epoch [3/20] Loss: 1.9349 | Acc: 45.83% | ε: 1.79
Epoch [4/20] Loss: 1.9782 | Acc: 48.53% | ε: 1.88
Epoch [5/20] Loss: 2.0649 | Acc: 50.14% | ε: 1.96
Epoch [6/20] Loss: 2.1313 | Acc: 50.82% | ε: 2.04
Epoch [7/20] Loss: 2.1406 | Acc: 51.13% | ε: 2.12
Epoch [8/20] Loss: 2.1695 | Acc: 52.36% | ε: 2.20
Epoch [9/20] Loss: 2.1975 | Acc: 52.87% | ε: 2.27
Epoch [10/20] Loss: 2.2625 | Acc: 53.24% | ε: 2.34
Epoch [11/20] Loss: 2.2345 | Acc: 53.25% | ε: 2.41
Epoch [12/20] Loss: 2.2503 | Acc: 53.22% | ε: 2.48
Epoch [13/20] Loss: 2.2561 | Acc: 54.02% | ε: 2.55
Epoch [14/20] Loss: 2.2336 | Acc: 54.10% | ε: 2.61
Epoch [15/20] Loss: 2.2777 | Acc: 53.88% | ε: 2.68
Epoch [16/20] Loss: 2.2652 | Acc: 53.47% | ε: 2.74
Epoch [17/20] Loss: 2.3179 | Acc: 53.53% | ε: 2.81
Epoch [18/20] Loss: 2.2758 | Acc: 54.03% | ε: 2.87
Epoch [19/20] Loss: 2.2500 | Acc: 53.72% | ε: 2.93
E

In [16]:
train_acc_dp3 = evaluate_accuracy(dp_model_3, train_loader_dp3, device)
test_acc_dp3 = evaluate_accuracy(dp_model_3, test_loader_dp3, device)

print(f"\nDP Model (ε=3) Performance:")
print(f"  Training accuracy: {train_acc_dp3:.2f}%")
print(f"  Test accuracy: {test_acc_dp3:.2f}%")
print(f"  Overfitting gap: {train_acc_dp3 - test_acc_dp3:.2f}%")
# Output: Training accuracy: 54.12%, Test accuracy: 53.01%, Overfitting gap: 1.11%

mia_acc_dp3, mia_adv_dp3 = compute_mia_advantage(
    dp_model_3, train_loader_dp3, test_loader_dp3, device
)

print(f"\nMIA Results (DP ε=3):")
print(f"  Attack accuracy: {mia_acc_dp3:.4f}")
print(f"  Attack advantage: {mia_adv_dp3:.4f}")
# Output: Attack accuracy: 0.5040, Attack advantage: 0.0040

improvement_3 = mia_adv_baseline - mia_adv_dp3
print(f"\nPrivacy Improvement vs Baseline:")
print(f"  MIA advantage reduction: {improvement_3:+.4f}")
print(f"  Accuracy cost: {test_acc_baseline - test_acc_dp3:.2f}%")
# Output: MIA advantage reduction: +0.0150, Accuracy cost: 14.12%



DP Model (ε=3) Performance:
  Training accuracy: 53.75%
  Test accuracy: 52.43%
  Overfitting gap: 1.32%

MIA Results (DP ε=3):
  Attack accuracy: 0.5064
  Attack advantage: 0.0064

Privacy Improvement vs Baseline:
  MIA advantage reduction: +0.0160
  Accuracy cost: 12.52%


In [17]:
def print_comparison_table(test_acc_baseline, test_acc_dp10, test_acc_dp3,
                           mia_adv_baseline, mia_adv_dp10, mia_adv_dp3,
                           final_epsilon_10, final_epsilon_3):
    """Print a comparison table of all model results."""
    print("\n" + "=" * 70)
    print("  COMPARISON TABLE")
    print("=" * 70)
    print(f"\n{'Model':<20} {'Test Acc':<12} {'MIA Adv':<12} {'Epsilon':<10}")
    print("-" * 55)
    print(f"{'Baseline':<20} {f'{test_acc_baseline:.2f}%':<12} {mia_adv_baseline:<12.4f} {'∞':<10}")
    print(f"{'DP (ε=10)':<20} {f'{test_acc_dp10:.2f}%':<12} {mia_adv_dp10:<12.4f} {final_epsilon_10:<10.2f}")
    print(f"{'DP (ε=3)':<20} {f'{test_acc_dp3:.2f}%':<12} {mia_adv_dp3:<12.4f} {final_epsilon_3:<10.2f}")

In [18]:
def save_results(test_acc_baseline, test_acc_dp10, test_acc_dp3,
                 mia_adv_baseline, mia_adv_dp10, mia_adv_dp3,
                 final_epsilon_10, final_epsilon_3, save_path="output/results.json"):
    """Save all results to a JSON file."""
    results = {
        "baseline": {
            "test_accuracy": float(test_acc_baseline),
            "mia_advantage": float(mia_adv_baseline),
        },
        "dp_eps10": {
            "test_accuracy": float(test_acc_dp10),
            "mia_advantage": float(mia_adv_dp10),
            "epsilon": float(final_epsilon_10),
        },
        "dp_eps3": {
            "test_accuracy": float(test_acc_dp3),
            "mia_advantage": float(mia_adv_dp3),
            "epsilon": float(final_epsilon_3),
        },
    }
    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)

In [19]:
print_comparison_table(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    mia_adv_baseline, mia_adv_dp10, mia_adv_dp3,
    final_epsilon_10, final_epsilon_3
)



  COMPARISON TABLE

Model                Test Acc     MIA Adv      Epsilon   
-------------------------------------------------------
Baseline             64.95%       0.0224       ∞         
DP (ε=10)            58.72%       0.0072       9.99      
DP (ε=3)             52.43%       0.0064       2.99      


In [20]:
plot_accuracy_comparison(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    save_path="figs/dp_sgd_accuracy_comparison.png"
)
print("Saved accuracy comparison to figs/dp_sgd_accuracy_comparison.png")

Saved accuracy comparison to figs/dp_sgd_accuracy_comparison.png


In [21]:
plot_privacy_utility_tradeoff(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    mia_adv_baseline, mia_adv_dp10, mia_adv_dp3,
    final_epsilon_10, final_epsilon_3,
    save_path="figs/dp_sgd_privacy_utility_tradeoff.png"
)
print("Saved privacy-utility tradeoff to figs/dp_sgd_privacy_utility_tradeoff.png")


Saved privacy-utility tradeoff to figs/dp_sgd_privacy_utility_tradeoff.png


In [22]:
save_results(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    mia_adv_baseline, mia_adv_dp10, mia_adv_dp3,
    final_epsilon_10, final_epsilon_3,
    save_path="output/results.json"
)
print("Results saved to output/results.json")

Results saved to output/results.json
